# 03 — Human Approval and Permissions

**Level:** Intermediate
**Time:** 2-3 hours

Welcome to Course 03. This course teaches how to build a **secure human approval subsystem** for consequential AI-agent actions.

**Scenario**: Northstar Cloud's checkout conversion has degraded in `eu-west` after deployment `deploy-1842`. The agent has gathered evidence and may propose a rollback, but it must **not** execute it directly. A human must authorize the exact scope of the action.

Let's establish our foundation first:
- **Authentication**: Who is the reviewer?
- **Authorization**: May this reviewer approve THIS type of action in THIS tenant/environment/risk tier?
- **Approval**: Did the reviewer approve THIS exact proposal?
- **Execution**: Can the approved action be executed safely once?
- **Audit**: Can we reconstruct proposal → decision → execution?


In [ ]:

# Setup our environment
import os
import sys

# Add our policy module to the path
sys.path.append(os.path.abspath("."))
sys.path.append(os.path.abspath("curriculum/intermediate/03-human-approval-permissions"))
from policy import (
    Service, Region, Environment, RiskTier, DecisionType,
    RollbackProposal, ReviewerContext, ApprovalPayload, ApprovalDecision,
    validate_approval, ApprovalStore, RollbackCommand, compute_risk, get_policy
)

from datetime import datetime, timezone, timedelta
import uuid

print("✅ Environment loaded. Ready to build secure approval pipelines.")



## Part 1: Authentication vs Authorization vs Approval

A model generated intent `is_approved = True` does not grant execution authority. The application owns the boundary. 

First, we define a **ReviewerContext**. This comes from your identity provider (e.g., Okta, Entra ID) and session, NOT the model.


In [ ]:

# The identity provider verified this session
alice_context = ReviewerContext(
    reviewer_id="alice",
    tenant_id="acme",
    roles={"incident_commander"},
    authenticated=True
)

bob_context = ReviewerContext(
    reviewer_id="bob",
    tenant_id="acme",
    roles={"operator"},
    authenticated=True
)

globex_eve = ReviewerContext(
    reviewer_id="eve",
    tenant_id="globex",
    roles={"incident_commander"},
    authenticated=True
)



## Part 2: Typed Rollback Proposal

The agent proposes business intent using a strict Pydantic model. It does not supply policy versions, tenant boundaries, or idempotency keys.


In [ ]:

# The agent generated this proposal after inspecting evidence
proposal = RollbackProposal(
    service=Service.CHECKOUT,
    region=Region.EU_WEST,
    deployment_id="deploy-1842",
    reason="Conversion dropped by 40% immediately after deploy",
    evidence_ids=["logs-456", "health-123"]
)

print(f"Agent proposes: Rollback {proposal.service.value} to {proposal.deployment_id} in {proposal.region.value}")



## Part 3: Approval Policy and Risk Tiers

Approval requirements depend on the action type, blast radius, environment, and tenant policy.
Let's see our deterministic risk classification.


In [ ]:

env = Environment.PRODUCTION
risk = compute_risk(proposal, env)

# Our application deterministic policy determines requirements based on risk
policy = get_policy("production_rollback", env, risk)

print(f"Risk Tier: {risk.value}")
print(f"Required Roles: {policy.required_roles}")
print(f"Approvals Required: {policy.approvals_required}")



## Part 4: Approval Payload and Digest Binding

The human must review a normalized, exact action. We create an `ApprovalPayload` that binds the proposal, tenant, policy version, and evidence. 

Most importantly, we compute a **SHA-256 digest**. The human approves this digest, not just the general idea of a rollback.


In [ ]:

payload = ApprovalPayload(
    proposal=proposal,
    tenant_id="acme",
    environment=Environment.PRODUCTION,
    risk_tier=risk,
    evidence_ids=["logs-456", "health-123"],
    policy_version="v3",
    expires_at=datetime.now(timezone.utc) + timedelta(hours=1)
)

print(f"Approval Digest: {payload.digest}")



## Part 5: Approve / Reject / Modify / Escalate

The human reviewer makes a decision on the exact digest.


In [ ]:

# Alice reviews the exact digest and approves it
decision_approve = ApprovalDecision(
    decision=DecisionType.APPROVE,
    approver_id="alice",
    approved_digest=payload.digest,
    reason="Confirmed metric drop, deploy-1842 is the culprit.",
    decided_at=datetime.now(timezone.utc)
)

# The authoritative validation converts the payload + decision into a trusted command
cmd = validate_approval(
    payload=payload,
    decisions=[decision_approve],
    reviewers=[alice_context],
    proposer_id="agent-01",
    current_policy_version="v3"
)

print(f"✅ Success! Created internal trusted command: {cmd.idempotency_key}")



What happens if Alice rejects it?


In [ ]:

decision_reject = ApprovalDecision(
    decision=DecisionType.REJECT,
    approver_id="alice",
    approved_digest=payload.digest,
    reason="Actually, the database is down. Do not rollback checkout.",
    decided_at=datetime.now(timezone.utc)
)

try:
    validate_approval(payload, [decision_reject], [alice_context], "agent-01", "v3")
except Exception as e:
    print(f"Blocked (Expected): {e}")



What if the human modifies the proposal? (e.g. they want to rollback GLOBAL instead of EU_WEST).
**Modifying an approved action creates a NEW action.** You must re-compute the digest and re-validate.


In [ ]:

# Human modifies region to GLOBAL
modified_proposal = proposal.model_copy(update={"region": Region.GLOBAL})
modified_payload = payload.model_copy(update={"proposal": modified_proposal})

print(f"Original Digest: {payload.digest}")
print(f"Modified Digest: {modified_payload.digest}")
print("The original approval digest no longer matches the modified payload.")



## Part 6: Reviewer Authorization & Tenant Isolation

What if Eve from Globex tries to approve Acme's rollback?


In [ ]:

try:
    validate_approval(payload, [decision_approve], [globex_eve], "agent-01", "v3")
except Exception as e:
    print(f"Blocked (Expected): {e}")



What if Bob the operator tries to approve a HIGH risk production rollback?


In [ ]:

try:
    validate_approval(payload, [decision_approve], [bob_context], "agent-01", "v3")
except Exception as e:
    print(f"Blocked (Expected): {e}")



## Part 7: Approval Expiry / Evidence Freshness

Approvals bind to time and state. If an approval expires, the validation fails.


In [ ]:

# Payload that expired 5 minutes ago
expired_payload = payload.model_copy(update={"expires_at": datetime.now(timezone.utc) - timedelta(minutes=5)})

try:
    validate_approval(expired_payload, [decision_approve], [alice_context], "agent-01", "v3")
except Exception as e:
    print(f"Blocked (Expected): {e}")



## Part 8: Application-owned Idempotency

The model does NOT generate UUIDs for consequential writes. The application derives a stable, logical idempotency key from the exact validated payload.


In [ ]:

store = ApprovalStore()

print(f"Application Derived Key: {cmd.idempotency_key}")

# First execution
receipt1 = store.record_execution(cmd, "EXECUTED")
print(f"First execution: {receipt1.status}")

# Replay (e.g. network timeout retry)
receipt2 = store.check_idempotency(cmd)
print(f"Replay execution: {receipt2.status}")



## Part 9: Unknown Execution Outcomes

If we retry the same idempotency key but the payload has mutated, it is a severe conflict.


In [ ]:

# Imagine an attacker captures the key and tries to attach it to a modified command
evil_cmd = cmd.model_copy(update={"approval_digest": "evil_digest"})

receipt3 = store.check_idempotency(evil_cmd)
print(f"Mutated replay execution: {receipt3.status}")



## Part 10: Audit Trail

Every action must be auditable. The `ApprovalStore` can record `ApprovalAuditEvent`s for every step (Propose, Approve, Execute).


In [ ]:

from policy import ApprovalAuditEvent

audit_event = ApprovalAuditEvent(
    event_id=str(uuid.uuid4()),
    run_id="run-999",
    tenant_id=cmd.tenant_id,
    proposal_digest=cmd.approval_digest,
    event_type="EXECUTION_STARTED",
    decision=None,
    reviewer_id=None,
    reviewer_roles=None,
    reason="Executing authorized command",
    policy_version=cmd.policy_version,
    timestamp=datetime.now(timezone.utc)
)

store.record_event(audit_event)
print(f"Recorded 1 audit event. Total events: {len(store.audit_events)}")



## Part 11: Two-person Approval & Separation of Duties

A CRITICAL rollback (e.g. GLOBAL checkout) requires 2 incident commanders, and the proposer cannot approve their own action.


In [ ]:

critical_proposal = proposal.model_copy(update={"region": Region.GLOBAL})
critical_risk = compute_risk(critical_proposal, Environment.PRODUCTION)

critical_payload = payload.model_copy(update={
    "proposal": critical_proposal,
    "risk_tier": critical_risk
})

alice_decision = decision_approve.model_copy(update={"approved_digest": critical_payload.digest})

# Try to validate with only 1 approval
try:
    validate_approval(critical_payload, [alice_decision], [alice_context], "agent-01", "v3")
except Exception as e:
    print(f"Blocked (Expected): {e}")

# Try to use Alice twice (SoD violation)
try:
    validate_approval(critical_payload, [alice_decision, alice_decision], [alice_context, alice_context], "agent-01", "v3")
except Exception as e:
    print(f"Blocked (Expected): {e}")

# Add a second authorized reviewer
charlie_context = ReviewerContext(
    reviewer_id="charlie",
    tenant_id="acme",
    roles={"incident_commander", "sre_lead"},
    authenticated=True
)
charlie_decision = ApprovalDecision(
    decision=DecisionType.APPROVE, approver_id="charlie",
    approved_digest=critical_payload.digest, reason="Concur",
    decided_at=datetime.now(timezone.utc)
)

critical_cmd = validate_approval(
    critical_payload, 
    [alice_decision, charlie_decision], 
    [alice_context, charlie_context], 
    "agent-01", "v3"
)
print(f"✅ Success! Two-person CRITICAL command created: {critical_cmd.idempotency_key}")



## Part 12: LangGraph Interrupt/Resume Adapter

LangGraph is fantastic for pausing and resuming workflows, but it is NOT the security boundary. The `interrupt()` function durably halts the agent, and when the workflow resumes, we perform our exact `validate_approval()` check.

Below is a mock implementation showing how the framework adapts to our deterministic security model.


In [ ]:

from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

class State(TypedDict):
    proposal: RollbackProposal
    payload: ApprovalPayload
    decision: ApprovalDecision
    command: RollbackCommand

def build_payload(state: State):
    proposal = state["proposal"]
    risk = compute_risk(proposal, Environment.PRODUCTION)
    payload = ApprovalPayload(
        proposal=proposal,
        tenant_id="acme",
        environment=Environment.PRODUCTION,
        risk_tier=risk,
        evidence_ids=proposal.evidence_ids,
        policy_version="v3",
        expires_at=datetime.now(timezone.utc) + timedelta(hours=1)
    )
    return {"payload": payload}

def human_review(state: State):
    payload = state["payload"]
    # Provide the EXACT typed review payload for the UI
    review_payload = {
        "digest": payload.digest,
        "service": payload.proposal.service.value,
        "region": payload.proposal.region.value,
        "deployment": payload.proposal.deployment_id,
        "risk": payload.risk_tier.value,
        "expires_at": str(payload.expires_at)
    }
    
    # ⏸️ PAUSE DURABLY
    decision = interrupt(review_payload)
    
    return {"decision": decision}

def execute(state: State):
    # ▶️ RESUME SECURITY VALIDATION
    payload = state["payload"]
    decision = state["decision"]
    
    # Normally ReviewerContext comes from the active web session performing the resume
    # Hardcoded here for the demo
    cmd = validate_approval(payload, [decision], [alice_context], "agent-01", "v3")
    
    print(f"Executing rollback command: {cmd.idempotency_key}")
    return {"command": cmd}

builder = StateGraph(State)
builder.add_node("build_payload", build_payload)
builder.add_node("human_review", human_review)
builder.add_node("execute", execute)
builder.add_edge(START, "build_payload")
builder.add_edge("build_payload", "human_review")
builder.add_edge("human_review", "execute")
builder.add_edge("execute", END)

# Note: MemorySaver is an in-memory checkpointer for local lab testing only.
# Production requires a durable database-backed checkpointer (e.g. PostgresSaver)
graph = builder.compile(checkpointer=MemorySaver())

print("✅ LangGraph orchestrated.")



In [ ]:

# Run the graph until the interrupt
thread_config = {"configurable": {"thread_id": "incident-123"}}

initial_state = {
    "proposal": RollbackProposal(
        service=Service.CHECKOUT, region=Region.EU_WEST, deployment_id="deploy-1842",
        reason="Drop", evidence_ids=["logs-1"]
    )
}

print("Starting graph...")
for event in graph.stream(initial_state, config=thread_config):
    pass

state = graph.get_state(thread_config)
print(f"State paused at: {state.next}")
print(f"Review Payload sent to UI: {state.tasks[0].interrupts[0].value}")



In [ ]:

# A human approves it via the UI, creating an ApprovalDecision
review_request = state.tasks[0].interrupts[0].value
human_decision = ApprovalDecision(
    decision=DecisionType.APPROVE,
    approver_id="alice",
    approved_digest=review_request["digest"],
    reason="Looks good.",
    decided_at=datetime.now(timezone.utc)
)

print("Resuming graph...")
for event in graph.stream(Command(resume=human_decision), config=thread_config):
    pass

print("✅ Workflow completed safely.")



## Part 13: Evaluation Harness

Let's test all failure modes and demonstrate that our policy is completely robust.


In [ ]:

scenarios = [
    ("Valid approval", payload, decision_approve, alice_context, "v3", None),
    ("Wrong tenant", payload, decision_approve, globex_eve, "v3", "WRONG_TENANT"),
    ("Unauthorized role", payload, decision_approve, bob_context, "v3", "UNAUTHORIZED_REVIEWER"),
    ("Expired approval", expired_payload, decision_approve, alice_context, "v3", "EXPIRED_APPROVAL"),
    ("Policy changed", payload, decision_approve, alice_context, "v4", "POLICY_CHANGED"),
    ("Reject decision", payload, decision_reject, alice_context, "v3", "DECISION_REJECT"),
]

print(f"{'Scenario':<30} | {'Expected':<25} | {'Actual':<25} | {'Passed'}")
print("-" * 90)

for name, p_load, dec, ctx, pol_ver, expected_err in scenarios:
    actual_err = None
    try:
        validate_approval(p_load, [dec], [ctx], "agent-01", pol_ver)
    except Exception as e:
        actual_err = str(e)
        
    passed = actual_err == expected_err
    status = "✅" if passed else "❌"
    
    print(f"{name:<30} | {str(expected_err):<25} | {str(actual_err):<25} | {status}")



## Part 14: Optional Real OpenAI Proposal Generation

We can use OpenAI to analyze evidence and generate the `RollbackProposal` object.
**The model stops there. It never approves or executes.** The proposal goes to the exact same deterministic human approval subsystem.


In [ ]:

import os
from pydantic import BaseModel
from openai import OpenAI

# (Optional) If you have an OpenAI API key, you can run this cell.
# Otherwise, it skips safely.
if os.getenv("OPENAI_API_KEY"):
    client = OpenAI()
    
    system_prompt = """
    You are an incident response agent. Analyze the evidence and output a RollbackProposal if necessary.
    """
    
    evidence_text = """
    Incident 991:
    - EU West Checkout latency spiked to 5000ms.
    - Deploy-1842 was released 10 minutes ago.
    """
    
    try:
        completion = client.beta.chat.completions.parse(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": evidence_text}
            ],
            response_format=RollbackProposal,
        )
        
        generated_proposal = completion.choices[0].message.parsed
        print(f"Generated Proposal from GPT-4o: {generated_proposal}")
    except Exception as e:
        print(f"OpenAI Generation Failed: {e}")
else:
    print("Skipping OpenAI demo (no OPENAI_API_KEY).")

